In [0]:
dbutils.secrets.listScopes()
dbutils.secrets.list("adls-scope")
client_secret = dbutils.secrets.get(scope="adls-scope", key="client-secret")
client_id = "16e41178-4311-45a7-b2ec-ff3999a7593b"
tenant_id = "0bf495b2-f9e4-4786-82ea-d1ff2c6f9353"


In [0]:
storage_account = "dlforformula1"
container = "bronze"

account_fqdn = f"{storage_account}.dfs.core.windows.net"

# (Optional but helpful) remove any key-based config that may be set on cluster
for k in [
    f"fs.azure.account.key.{account_fqdn}",
    f"fs.azure.account.keyprovider.{account_fqdn}",
    "fs.azure.account.key"
]:
    try:
        spark.conf.unset(k)
    except:
        pass

# Set OAuth configs (IMPORTANT: include the account FQDN in the key)
spark.conf.set(f"fs.azure.account.auth.type.{account_fqdn}", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{account_fqdn}",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{account_fqdn}", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{account_fqdn}",
               dbutils.secrets.get(scope="adls-scope", key="client-secret"))
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{account_fqdn}",
               f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Now try listing
path = f"abfss://{container}@{account_fqdn}/"
display(dbutils.fs.ls(path))

In [0]:
df = spark.range(0, 1000000)
df.groupBy().count().show()

In [0]:
df = spark.range(0, 1000000)
df.groupBy("id").count().show()

In [0]:
df = spark.range(0, 1000000)
df = df.withColumn("key", df.id % 10)
from pyspark.sql.functions import monotonically_increasing_id
df  = df.withColumn("col1", monotonically_increasing_id())

# df = df.withColumn("col2", monotonically_increasing_id()
df.groupBy("col1", "key").count().show()

## ✅ Format of Storage Account Location Using `abfss://`

When accessing **ADLS Gen2** from **Databricks or Spark**, the secure protocol used is:

> `abfss://` → Azure Blob File System Secure (SSL enabled)

---

# 🔷 General Format

```text
abfss://<container>@<storage-account>.dfs.core.windows.net/<path>
```

---

# 🔷 Breakdown of Each Part

| Part                    | Meaning                              |
| ----------------------- | ------------------------------------ |
| `abfss://`              | Secure ADLS Gen2 protocol            |
| `<container>`           | ADLS container name                  |
| `<storage-account>`     | Your storage account name            |
| `.dfs.core.windows.net` | ADLS Gen2 endpoint                   |
| `<path>`                | Folder or file path inside container |

---

# 🔷 Example

If:

* Storage Account → `dlforformula1`
* Container → `bronze`
* Folder → `race`
* File → `race.csv`

Then the full path is:

```text
abfss://bronze@dlforformula1.dfs.core.windows.net/race/race.csv
```

---

# 🔷 Example in Databricks (Spark)

Using **Azure Databricks**:

```python
df = spark.read.csv(
    "abfss://bronze@dlforformula1.dfs.core.windows.net/race/race.csv"
)
```

---

# 🔷 For Writing Delta

```python
df.write.format("delta").mode("append").save(
    "abfss://bronze@dlforformula1.dfs.core.windows.net/silver/race"
)
```

---

# 🔷 Important Notes

### ✅ Use `.dfs.core.windows.net`

For ADLS Gen2 (Hierarchical Namespace enabled)

### ❌ Do NOT use `.blob.core.windows.net`

That is for Blob Storage endpoint (used with AzCopy or SAS)

---

# 🔐 Production Standard

With **Unity Catalog**, you usually don’t directly use `abfss://` paths in production.
Instead, you use:

```sql
catalog.schema.table
```

But internally, Unity Catalog still points to an `abfss://` external location.

---

If you want, I can also explain:

* Difference between `abfs` vs `abfss`
* Difference between `dfs.core` vs `blob.core`
* How authentication works behind `abfss`
* How this maps to AWS S3 path format


In [0]:
# abfss://demo@dlforformula1.dfs.core.windows.net/circuits.csv
# format: abfss://<container>@<storage-account>.dfs.core.windows.net/<path>
spark.read.csv("abfss://demo@dlforformula1.dfs.core.windows.net/circuits.csv")